In [31]:
excel_path = '/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/excel.xlsx'
video_path = '/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/fifteen_twenty_five'

In [32]:
import os
import cv2
import numpy as np
import pandas as pd
from keras.applications import ResNet50
from keras.models import Model
from keras.layers import Conv2D, LSTM, Dense, Flatten, Concatenate, Input
from keras.optimizers import Adam
from moviepy.editor import VideoFileClip
import librosa
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

# ----------------------------------------------
# Data Ingestion
# ----------------------------------------------
def load_metadata(metadata_path):
    """
    Load metadata from Excel file.
    """
    return pd.read_excel(metadata_path)

# ----------------------------------------------
# Preprocessing - Video and Audio Feature Extraction
# ----------------------------------------------

def sample_frames(video_path, n_frames=30, resize_shape=(224, 224)):
    """
    Sample `n_frames` evenly from the video file, resizing each frame.
    """
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    
    for i in range(n_frames):
        frame_idx = int(frame_count * i / n_frames)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, resize_shape)
            frames.append(frame)
    
    cap.release()
    return np.array(frames)

def extract_audio(video_path, audio_sample_rate=22050, n_mfcc=13):
    """
    Extract MFCC audio features from the video.
    """
    clip = VideoFileClip(video_path)
    audio = clip.audio
    audio_path = video_path.replace(".mp4", ".wav")
    audio.write_audiofile(audio_path)
    
    y, sr = librosa.load(audio_path, sr=audio_sample_rate)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return mfcc

# ----------------------------------------------
# Metadata Encoding
# ----------------------------------------------
def encode_metadata(metadata):
    """
    Encode categorical metadata like laughter and joke type.
    """
    le = LabelEncoder()
    metadata['Laugh_encoded'] = le.fit_transform(metadata['Laugh'])
    metadata['HowFunny_encoded'] = le.fit_transform(metadata['JokeType'])
    metadata['HowFunny_encoded'] = le.fit_transform(metadata['HowFunny'])
    return metadata

# ----------------------------------------------
# Feature Extraction Using Pre-Trained Models
# ----------------------------------------------
def extract_visual_features(frames, model):
    """
    Extract visual features from video frames using a pre-trained CNN model.
    """
    features = [model.predict(np.expand_dims(frame, axis=0)) for frame in frames]
    return np.array(features)

# ----------------------------------------------
# Model Development
# ----------------------------------------------
def build_multimodal_model(input_shape_video, input_shape_audio):
    """
    Build a multi-modal model combining video (CNN) and audio (RNN) features.
    """
    # Video input
    video_input = Input(shape=input_shape_video)
    x1 = Conv2D(32, (3, 3), activation='relu')(video_input)
    x1 = Flatten()(x1)

    # Audio input
    audio_input = Input(shape=input_shape_audio)
    x2 = LSTM(64)(audio_input)

    # Concatenate video and audio features
    combined = Concatenate()([x1, x2])

    # Output layer
    output = Dense(4, activation='softmax')(combined)

    # Compile the model
    model = Model(inputs=[video_input, audio_input], outputs=output)
    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------
# Training and Evaluation
# ----------------------------------------------
def train_model(model, video_features, audio_features, labels, batch_size=16, epochs=10):
    """
    Train the multi-modal model.
    """
    return model.fit([video_features, audio_features], labels, 
                     batch_size=batch_size, epochs=epochs, validation_split=0.2)

def evaluate_model(model, test_video_features, test_audio_features, test_labels):
    """
    Evaluate the model on test data.
    """
    predictions = model.predict([test_video_features, test_audio_features])
    accuracy = accuracy_score(test_labels, np.argmax(predictions, axis=1))
    f1 = f1_score(test_labels, np.argmax(predictions, axis=1), average='weighted')
    cm = confusion_matrix(test_labels, np.argmax(predictions, axis=1))
    return accuracy, f1, cm

# ----------------------------------------------
# Visualization
# ----------------------------------------------
def plot_confusion_matrix(cm, labels):
    """
    Plot confusion matrix.
    """
    fig, ax = plt.subplots(figsize=(8, 8))
    cax = ax.matshow(cm, cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    fig.colorbar(cax)
    ax.set_xticklabels([''] + labels)
    ax.set_yticklabels([''] + labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

# ----------------------------------------------
# Main Implementation Workflow
# ----------------------------------------------
def main(video_dir, metadata_path):
    """
    Main workflow for the entire pipeline.
    """
    # Step 1: Load metadata
    metadata = load_metadata(metadata_path)
    metadata = encode_metadata(metadata)

    # Step 2: Initialize pre-trained model for visual feature extraction
    cnn_model = ResNet50(weights='imagenet', include_top=False)

    video_features = []
    audio_features = []
    labels = []

    # Step 3: Loop through each video and extract features
    for idx, row in metadata.iterrows():
        video_path = os.path.join(video_dir, f"{row['VideoID']}")
        frames = sample_frames(video_path)
        audio_mfcc = extract_audio(video_path)

        # Extract video features using CNN
        visual_features = extract_visual_features(frames, cnn_model)
        video_features.append(visual_features)
        
        # Store audio and labels
        audio_features.append(audio_mfcc)
        labels.append(row['HowFunny_encoded'])

    # Convert lists to arrays
    video_features = np.array(video_features)
    audio_features = np.array(audio_features)
    labels = np.array(labels)

    # Step 4: Split data into training and testing sets
    X_train_video, X_test_video, X_train_audio, X_test_audio, y_train, y_test = train_test_split(
        video_features, audio_features, labels, test_size=0.2, random_state=42
    )

    # Step 5: Build the multi-modal model
    input_shape_video = video_features.shape[1:]  # Shape of video features
    input_shape_audio = audio_features.shape[1:]  # Shape of audio features
    model = build_multimodal_model(input_shape_video, input_shape_audio)

    # Step 6: Train the model
    train_model(model, X_train_video, X_train_audio, y_train)

    # Step 7: Evaluate the model
    accuracy, f1, cm = evaluate_model(model, X_test_video, X_test_audio, y_test)

    print(f"Accuracy: {accuracy}")
    print(f"F1 Score: {f1}")
    
    # Step 8: Plot confusion matrix
    plot_confusion_matrix(cm, labels=metadata['HowFunny'].unique())

# Example usage
if __name__ == "__main__":
    
    main(video_path, excel_path)


OpenCV: Couldn't read video stream from file "/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/fifteen_twenty_five/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4"


OSError: MoviePy error: the file /Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/dataa/fifteen_twenty_five/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4 could not be found!
Please check that you entered the correct path.

In [17]:
pwd

'/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october'